# GroupBy Operations

---

### Table of Contents
1. Introduction to "Group By"
2. Performing a Simple Group By
3. Grouping and Selecting a Single Column
4. Grouping By Multiple Columns
5. The `.agg()` Method for Multiple Aggregations

---

## 1. Introduction to "Group By"
- The "group by" operation is one of the most powerful features in Pandas.
- It is used for splitting data into groups based on some criteria, applying
  a function to each group independently, and then combining the results.
- This process is often referred to as "Split-Apply-Combine"

In [8]:
import os
import pandas as pd

# --- Setup: Load the data and create a 'TotalPrice' column ---
DATA_FOLDER = "pandas_data"
file_path = os.path.join(DATA_FOLDER, "sample_sales_data.csv")

try:
    df = pd.read_csv(file_path, parse_dates=["OrderDate"])
    # Create a TotalPrice column for more meaningful aggregations
    df["TotalPrice"] = df["Price"] * df["Quantity"]
    print("--- Sample DataFrame with TotalPrice ---")
    print(df)
except FileNotFoundError:
    print(f"Error: The data file was not found at '{file_path}'")
    print("Please run '04_reading_and_writing_data.py' first to create it.")
    df = pd.DataFrame()  # Create an empty df to avoid further errors

--- Sample DataFrame with TotalPrice ---
   OrderID   Product     Category   Price  Quantity  OrderDate  TotalPrice
0      101    Laptop  Electronics  1200.0         1 2025-01-15      1200.0
1      102     Mouse  Electronics    25.5         2 2025-01-15        51.0
2      103  Keyboard  Electronics    75.0         1 2025-01-16        75.0
3      104   Monitor  Electronics   300.0         2 2025-01-17       600.0
4      105     Mouse  Accessories    27.0         3 2025-01-18        81.0
5      106    Webcam  Accessories    50.0         1 2025-01-18        50.0



---

## 2. Performing a Simple Group By
- The `.groupby()` method itself doesn't compute anything; it returns a `DataFrameGroupBy` object.
- This object contains all the information about the groups. To get a result,
  you must chain an aggregation method to it (e.g., .sum(), .mean()).

In [9]:
# Group the DataFrame by the 'Category' column
category_groups = df.groupby("Category")

# Now, apply aggregation functions to the groups:

In [10]:
# This calculates the sum of all numerical columns for each category
category_sum = category_groups.sum(numeric_only=True)
print("Sum of numerical columns by Category:\n", category_sum)

Sum of numerical columns by Category:
              OrderID   Price  Quantity  TotalPrice
Category                                          
Accessories      211    77.0         4       131.0
Electronics      410  1600.5         6      1926.0


In [11]:
# Calculate the mean of all numerical columns for each category
category_mean = category_groups.mean(numeric_only=True)
print("\nMean of numerical columns by Category:\n", category_mean)


Mean of numerical columns by Category:
              OrderID    Price  Quantity  TotalPrice
Category                                           
Accessories    105.5   38.500       2.0        65.5
Electronics    102.5  400.125       1.5       481.5


In [12]:
# .size() counts the number of rows in each group
category_size = category_groups.size()
print("\nSize (number of items) of each Category:\n", category_size)


Size (number of items) of each Category:
 Category
Accessories    2
Electronics    4
dtype: int64



---

## 3. Grouping and Selecting a Single Column
- For efficiency, you can select a column *before* aggregating.
- This is useful when you only care about the statistics of one specific column.

In [13]:
# Calculate the total revenue ('TotalPrice') for each product
# The result is a Pandas Series
product_revenue = df.groupby("Product")["TotalPrice"].sum()
print("Total revenue per Product:\n", product_revenue)

Total revenue per Product:
 Product
Keyboard      75.0
Laptop      1200.0
Monitor      600.0
Mouse        132.0
Webcam        50.0
Name: TotalPrice, dtype: float64


In [14]:
# Find the average price for each category
avg_price_per_category = df.groupby("Category")["Price"].mean()
print("\nAverage price per Category:\n", avg_price_per_category)


Average price per Category:
 Category
Accessories     38.500
Electronics    400.125
Name: Price, dtype: float64



---

## 4. Grouping By Multiple Columns
- You can pass a list of column names to group by multiple levels.
- This results in a DataFrame with a MultiIndex.

In [15]:
# Group by both 'Category' and 'Product'
multi_group = df.groupby(["Category", "Product"])

In [16]:
# Calculate the total quantity sold for each combination
quantity_by_group = multi_group["Quantity"].sum()
print("Total quantity sold by Category and Product:\n", quantity_by_group)

Total quantity sold by Category and Product:
 Category     Product 
Accessories  Mouse       3
             Webcam      1
Electronics  Keyboard    1
             Laptop      1
             Monitor     2
             Mouse       2
Name: Quantity, dtype: int64



---

## 5. The `.agg()` Method for Multiple Aggregations
- The `.agg()` method is a flexible way to apply multiple aggregation functions at once.
- You can apply different functions to different columns.

In [17]:
# Group by category and apply multiple aggregations
aggregations = df.groupby("Category").agg(
    # Pass a dictionary where keys are columns and values are the functions to apply
    {
        "Price": "mean",  # Calculate the mean of the Price column
        "Quantity": "sum",  # Calculate the sum of the Quantity column
        "TotalPrice": [
            "sum",
            "max",
        ],  # Apply multiple functions to the TotalPrice column.
    }
)
print("Multiple aggregations by Category:\n", aggregations)

Multiple aggregations by Category:
                Price Quantity TotalPrice        
                mean      sum        sum     max
Category                                        
Accessories   38.500        4      131.0    81.0
Electronics  400.125        6     1926.0  1200.0



---

**Next:** [Merging, Joining, and Concatenating](./10_merging_joining_and_concatenating.ipynb)